# Getting Started with the Istari Fluent Client

This notebook walks through a first end-to-end experience against the Istari platform using [`istari_fluent`](../fluent) — the opinionated, chainable wrapper over the official `istari-digital-client`.

**What you'll do:**

1. Connect to an Istari environment using your `.env` credentials.
2. Find a System by name, creating it if it doesn't exist.
3. Upload `Group3-UAS-Requirements.xlsx` as a Model on that system.
4. Run an extraction job (`@istari:extract` / `open_spreadsheet`) and inspect its products.
5. Pick one product and chain it as a source into a second job.
6. Trace the backward lineage of the final output.

### Prerequisites

**1. Credentials** &mdash; copy `fluent/.env.example` into `samples/.env` and fill in:

```
ISTARI_ENVIRONMENT_URL=https://fileservice-v2.demo.istari.app
ISTARI_PAT=your-personal-access-token
```

The notebook will look for `samples/.env` first, then fall back to `fluent/.env`.

**2. Install dependencies** &mdash; from the repo root:

```bash
cd fluent
uv sync --extra experiment
```

This creates `fluent/.venv/` with `istari_fluent`, `jupyter`, and `ipykernel` installed.

**3. Register the venv as a Jupyter kernel** &mdash; a bare `.venv` is not auto-discovered; register it once with a friendly name:

```bash
# still inside fluent/
uv run python -m ipykernel install --user --name istari-fluent --display-name "Python (istari_fluent)"
```

Now reload VS Code / Cursor's kernel picker (top-right of this notebook) and select **"Python (istari_fluent)"**.

To remove it later: `jupyter kernelspec uninstall istari-fluent`.

> `istari_fluent` is a productivity layer maintained alongside the official SDK. It is not the officially supported client &mdash; for production integrations, keep the core `istari-digital-client` as your source of truth.

## 1. Connect to the platform

`IstariPlatform.from_env()` reads `ISTARI_ENVIRONMENT_URL` and `ISTARI_PAT` from your `.env` file.

In [ ]:
from pathlib import Path

from istari_fluent import IstariPlatform, JobDefinition

# Look for .env in samples/ first, then fall back to fluent/.env
_candidates = [
    Path.cwd() / ".env",
    Path.cwd().parent / "fluent" / ".env",
]
env_path = next((p for p in _candidates if p.exists()), None)
if env_path is None:
    raise FileNotFoundError(
        f"No .env found. Looked in: {[str(p) for p in _candidates]}"
    )
print(f"Using credentials from: {env_path}")

platform = IstariPlatform.from_env(dotenv_path=str(env_path))
platform

## 2. Find or create a system

`get_or_create_system` returns a `SystemView` either way — it looks up by name, and if nothing matches it creates a new System on the platform.

In [ ]:
SYSTEM_NAME = "UAS-Tutorial"

system = platform.get_or_create_system(
    SYSTEM_NAME,
    description="Group 3 UAS tutorial workspace (istari_fluent getting started)",
)
print(system)
print("Baseline configuration:", system.baseline.configuration.name)

## 3. Upload the spreadsheet as a Model

The UAS requirements file is already in this `samples/` folder.
`upload_model` creates a new Model on the platform and returns a `ModelView`.

We tag it with a stable `external_id` so re-running this notebook will find the existing Model via `find_model(external_id=...)` instead of creating duplicates.

In [ ]:
XLSX_PATH = Path.cwd() / "Group3-UAS-Requirements.xlsx"
EXTERNAL_ID = "fluent-tutorial-uas-requirements"

model = platform.find_model(external_id=EXTERNAL_ID)
if model is None:
    model = platform.upload_model(
        XLSX_PATH,
        external_id=EXTERNAL_ID,
        display_name="Group3-UAS-Requirements (tutorial)",
    )
    print("Uploaded new model.")
else:
    print("Reusing existing model.")

print(model)

## 4. Run the first extraction job

`@istari:extract` with the `open_spreadsheet` tool parses an Excel workbook and writes a handful of products:

- `named_cells.json` — named-range contents
- `worksheet_data.json` — cell data per sheet
- `workbook.pdf` / `workbook.html` — rendered views

`run_job` submits the job and blocks until it reaches a terminal state (or raises on failure / timeout).

In [ ]:
extract = JobDefinition(
    function="@istari:extract",
    tool_name="open_spreadsheet",
)

job1 = model.run_job(extract, timeout=600)
print(f"Job 1: {job1.status}  id={job1.id}")

## 5. Inspect the products

`job1.get_products()` reads `job.revision.products` — each result is a `ResourceView` **pinned to the exact revision the agent wrote**, so later jobs that touch the same files cannot change what we see here (race-safe).

In [ ]:
products_1 = job1.get_products()
print(f"Job 1 wrote {len(products_1)} products:\n")
for p in products_1:
    print(f"  - {p.type:10s}  name={p.name!r:30s}  file={p.file_id}  rev={p.revision_id}")

## 6. Pick a product and download it

`find_product` grabs a single pinned `ResourceView` by filename. From there you can stream the bytes, decode text, or download to disk.

In [ ]:
import json

named_cells = job1.find_product(filename="named_cells.json")
assert named_cells is not None, "named_cells.json not produced"

data = json.loads(named_cells.read_text())
print(f"named_cells.json has {len(data)} named ranges")
print("First 3 keys:", list(data)[:3])

download_path = named_cells.download(Path.cwd() / "outputs")
print(f"Downloaded to: {download_path}")

## 7. Chain a second job

Now we'll run **the same extraction a second time**, but feed one of Job 1's products in as an explicit source via `as_source()`. The platform records that link in the lineage graph, so later we can trace exactly which revision Job 2 consumed.

`as_source()` uses the product's `revision_id` directly (already on the `Product` record), so this does **not** cost an extra API call.

In [ ]:
named_cells_source = named_cells.as_source(relationship_identifier="input")
print("Attaching source:", named_cells_source)

job2 = model.run_job(
    extract,
    sources=[named_cells_source],
    timeout=600,
)
print(f"Job 2: {job2.status}  id={job2.id}")

products_2 = job2.get_products()
print(f"\nJob 2 wrote {len(products_2)} products:")
for p in products_2:
    print(f"  - {p.type:10s}  name={p.name!r:30s}  rev={p.revision_id}")

## 8. Trace the lineage

`get_lineage()` walks backward from a revision, classifying each step as one of:

- `upload` &mdash; a fresh file (no sources)
- `job_run` &mdash; produced by a Job
- `promotion` &mdash; a Model promoted from another revision (tag `promoted_from`)
- `derived` &mdash; any other derivation

Below we pick one of Job 2's products and print the chain back to the original upload.

In [ ]:
final_output = job2.find_product(filename="named_cells.json")
assert final_output is not None

tree = final_output.get_lineage(max_depth=6)
print("Lineage for Job 2's named_cells.json:\n")
tree.print_tree()

You can also iterate the tree flat, e.g. to find the originating upload or count how many job runs sit in the chain.

In [ ]:
from collections import Counter

steps = Counter(node.step for node in tree.walk())
print("Step counts:", dict(steps))

uploads = [n for n in tree.walk() if n.step == "upload"]
for u in uploads:
    print(f"Origin upload: {u.resource_type} {u.label!r} (rev={u.revision_id})")

## Recap

- `IstariPlatform.from_env()` &mdash; connect using `.env`.
- `platform.get_or_create_system(name)` &mdash; idempotent system lookup.
- `platform.upload_model(path, external_id=...)` / `platform.find_model(external_id=...)` &mdash; idempotent model registration.
- `model.run_job(definition)` &mdash; submit + wait, returns a completed `JobView`.
- `job.get_products()` / `job.find_product(filename=...)` &mdash; race-safe, pinned `ResourceView`s.
- `product.as_source()` &mdash; build a `NewSource` for chaining without an API call.
- `view.get_lineage().print_tree()` &mdash; trace provenance backward.

**Next steps**

- Try `artifact.run_job(...)` &mdash; the Artifact is auto-promoted to a Model before the job submits, so you can chain jobs on artifacts without an explicit `promote()` call.
- Explore `ResourceView.promote()` if you want a reusable, standalone Model out of a product.
- Use `TrackedFileSet` and `ConfigurationView.add_file(...).save()` to evolve a system's baseline through configuration versions.